# Anima 生图 + Wan 视频 - Colab 一次成型版

**这个版本把之前所有踩坑固化成自动逻辑，正常情况跑 1→2→3→4 四格就出图，不用再改任何东西。**

| 已固化的坑 | 处理方式 |
|---|---|
| T4 无 bf16，fp16 T5 溢出 -> 100% 黑图 | cell 3 自动测试：hybrid(transformer fp16) -> 黑图自动回退 fp32，**测过才锁** |
| `load_components(torch_dtype=)` 被新版 diffusers 忽略 | 只给 `from_pretrained(dtype=)` 传（新版关键字），load_components 不传 |
| 832x1216 在 fp32 下 OOM | 服务端 OOM 自动降档重试（x0.8 缩，最多 3 次） |
| 事后 .half()/搬 CPU -> 黑图/device 错乱 | 不存在了：CPU 上一次性定精度再上 GPU |
| 90 分钟无操作断线 | cell 2 自动保活 |
| /content 会话结束清空 | cell 2 挂 Drive，cell 4 有 /flush |
| 上次 RAM 爆掉 | 全程只保留一份 pipe，重载先 del + gc |

**预期速度（640x960@22步，白衬衫同款配置）**：hybrid ~25-40s / fp32 ~90-130s。
**无审查**：裸权重，想画什么写什么。

In [ ]:
#@title 1. 环境探测 + 依赖（先跑这格）
import os, sys, subprocess, shutil
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # 抗显存碎片

IN_COLAB = os.path.isdir('/content')
print('colab:', IN_COLAB)

HAS_GPU = False; GPU_NAME = 'none'; VRAM_GB = 0.0
try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                       capture_output=True, text=True, timeout=30)
    if r.returncode == 0 and r.stdout.strip():
        GPU_NAME = r.stdout.split(',')[0].strip()
        VRAM_GB = float(r.stdout.split(',')[1].replace('MiB','').strip()) / 1024
        HAS_GPU = True
except Exception as e:
    print('nvidia-smi:', e)
print('GPU : %s %.1fGB' % (GPU_NAME, VRAM_GB) if HAS_GPU else 'GPU : 无（CPU 档 -> OpenVINO 生图）')

try:
    import psutil
    vm = psutil.virtual_memory()
    print('RAM : %.1f 总 / %.1f 空闲' % (vm.total/1024**3, vm.available/1024**3))
except ImportError:
    pass
print('disk: %.1fGB free' % (shutil.disk_usage('/content' if IN_COLAB else '.').free/1024**3))

if IN_COLAB:
    %pip -q install diffusers transformers accelerate safetensors flask flask-cors psutil pillow requests numpy
    if not HAS_GPU:
        %pip -q install optimum-intel openvino

import torch
if HAS_GPU:
    torch.backends.cuda.matmul.allow_tf32 = True   # T4 fp32 免费提速 ~2x
    torch.backends.cudnn.allow_tf32 = True
print('[+] 就绪 | torch', torch.__version__)

In [ ]:
#@title 2. 会话保活 + Drive 落盘
import os, time, shutil, threading

OUT_DIR = '/content/out' if IN_COLAB else './out'
os.makedirs(OUT_DIR, exist_ok=True)

if IN_COLAB:
    from IPython.display import display, Javascript
    display(Javascript("""
    function ka(){var b=document.querySelector('colab-connect-button');
      if(b&&b.shadowRoot){var c=b.shadowRoot.querySelector('#connect');if(c)c.click();}}
    setInterval(ka,60000);"""))
    print('[+] keepalive 60s')

DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/ai_out'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DRIVE_OK = True
    print('[+] Drive ->', DRIVE_DIR)
except Exception as e:
    print('[!] 无 Drive（用浏览器下载也行）:', str(e)[:120])

def flush_to_drive(verbose=True):
    if not DRIVE_OK:
        if verbose: print('[!] Drive 不可用，产出在', OUT_DIR)
        return
    tgt = os.path.join(DRIVE_DIR, time.strftime('%m%d_%H%M'))
    os.makedirs(tgt, exist_ok=True)
    n = 0
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            shutil.copy2(os.path.join(root, f), os.path.join(tgt, f)); n += 1
    if verbose: print('[+] 搬了 %d 个文件 -> %s' % (n, tgt))

In [ ]:
#@title 3. 加载 Anima：自动选精度（黑图自动回退，测过才锁）
# 实测结论：全fp16 -> T5 溢出黑图；全fp32 -> 真图但慢；hybrid(transformer fp16+文本侧 fp32) 是速度解。
# 本格按 hybrid -> fp32 顺序逐个加载并出 512x512 测试图，mean<6 判黑自动换下一种，锁死能出真图的档。
MODEL_ID = 'circlestone-labs/Anima-Base-v1.0-Diffusers'
pipe = None
ANIMA_MODE = None

if not HAS_GPU:
    print('[!] CPU 档：生图走 OpenVINO 预设（cell 4 自动处理，本格跳过）')
else:
    import torch, gc, numpy as np
    if 'pipe' in globals() and pipe is not None:
        del pipe
    gc.collect(); torch.cuda.empty_cache()
    from diffusers import ModularPipeline

    def _mean(img):
        return float(np.asarray(img.convert('RGB')).mean())

    def _build(hybrid):
        global pipe
        if 'pipe' in globals() and pipe is not None:
            del pipe
        gc.collect(); torch.cuda.empty_cache()
        # 新版 diffusers 认 dtype=（torch_dtype 已弃用且被 load_components 忽略），旧版回退
        try:
            pipe = ModularPipeline.from_pretrained(MODEL_ID, dtype=torch.float16)
        except TypeError:
            pipe = ModularPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
        pipe.load_components()
        promote = ('text_encoder','text_conditioner') if hybrid else \
                  ('text_encoder','text_conditioner','transformer','vae')
        for a in promote:
            c = getattr(pipe, a, None)
            if c is not None and hasattr(c, 'to'):
                c.to(dtype=torch.float32)      # CPU 上定精度，零碎片
        pipe.to('cuda')
        pipe.set_progress_bar_config(disable=True)
        torch.cuda.empty_cache()

    def _test(tag):
        out = pipe(prompt='masterpiece, best quality, 1girl, portrait',
                   negative_prompt='worst quality', height=512, width=512,
                   num_inference_steps=8, generator=torch.Generator('cuda').manual_seed(7))
        m = _mean(out.images[0])
        ok = m >= 6
        print('  [%s] 测试图 mean=%.1f -> %s' % (tag, m, '真图 OK' if ok else '黑图'))
        return ok

    for mode, hyb in (('hybrid', True), ('fp32', False)):
        try:
            _build(hyb)
            if _test(mode):
                ANIMA_MODE = mode
                break
            print('  [%s] 黑图，自动试下一种精度' % mode)
        except Exception as e:
            print('  [%s] 异常: %s' % (mode, str(e)[:140]))

    free_b, total_b = torch.cuda.mem_get_info()
    if ANIMA_MODE:
        print('[+] Anima 锁定 mode=%s | 显存空闲 %.1f/%.1f GB' % (ANIMA_MODE, free_b/1024**3, total_b/1024**3))
        print('    640x960@22 预期: hybrid ~25-40s | fp32 ~90-130s')
    else:
        print('[x] 两种精度都失败，看上面报错重跑本格')

In [ ]:
#@title 4. 启动服务 + 隧道（DSH / CLI 直调入口）
import threading, uuid, os, io, json, time, re, queue, subprocess
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import numpy as np

app = Flask(__name__); CORS(app)
JOBS = {}; LOCK = threading.Lock(); TASK_Q = queue.Queue()
URL = None

# ---- CPU 档预设（无 GPU 才用）----
CPU_PRESETS = {
    'sana06':         {'model':'HelloSun/Sana_Sprint_0.6B_1024px-OpenVINO-INT4','steps':4,'guidance':4.5,'maxdim':1024,'neg':False,'extra':{'intermediate_timesteps':None}},
    'sdxl_lightning': {'model':'HelloSun/dreamshaper-xl-lightning-OpenVINO-INT4','steps':4,'guidance':2.0,'maxdim':768,'neg':True,'extra':{}},
    'sdxl_anime':     {'model':'HelloSun/nova_xl_lcm-OpenVINO-INT4','steps':8,'guidance':1.5,'maxdim':1024,'neg':True,'extra':{}},
    'animagine':      {'model':'HelloSun/animagine-xl-4.0-Lightning-OpenVINO-INT4','steps':4,'guidance':0.0,'maxdim':1024,'neg':False,'extra':{}},
    'sd15_lcm':       {'model':'HelloSun/animemixv10_lcm-OpenVINO-INT4','steps':8,'guidance':1.5,'maxdim':512,'neg':True,'extra':{}},
}
CPU_PRESET = 'sana06'
CPU = {'pipe': None, 'key': None, 'loading': False, 'error': None}

def set_cpu_preset(name):
    global CPU_PRESET
    if name in CPU_PRESETS and name != CPU_PRESET:
        CPU_PRESET = name; CPU['pipe'] = None; CPU['key'] = None; CPU['error'] = None

def load_cpu_pipe():
    if CPU['pipe'] is not None and CPU['key'] == CPU_PRESET:
        return CPU['pipe']
    if CPU['loading']:
        raise RuntimeError('CPU model loading, retry in 120s')
    CPU['loading'] = True; CPU['error'] = None
    try:
        from optimum.intel import OVDiffusionPipeline
        p = OVDiffusionPipeline.from_pretrained(CPU_PRESETS[CPU_PRESET]['model'], compile=True)
        CPU['pipe'] = p; CPU['key'] = CPU_PRESET
        return p
    except Exception as e:
        CPU['error'] = '%s: %s' % (type(e).__name__, e); raise
    finally:
        CPU['loading'] = False

def set_job(jid, **kw):
    with LOCK:
        JOBS.setdefault(jid, {'status':'queued','created':time.time()}).update(kw)

def get_job(jid):
    with LOCK:
        j = JOBS.get(jid)
        return dict(j) if j else None

# ---- 视频（懒加载，第一次请求才下 ~10-20GB）----
VID_MODEL = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
VIDEO = {'pipe': None, 'loading': False, 'error': None}

def load_video_pipe():
    if VIDEO['pipe'] is not None:
        return VIDEO['pipe']
    if VIDEO['loading']:
        raise RuntimeError('video model loading, retry in 60s')
    VIDEO['loading'] = True
    try:
        import torch
        from diffusers import WanPipeline, AutoencoderKLWan
        vae = AutoencoderKLWan.from_pretrained(VID_MODEL, subfolder='vae', torch_dtype=torch.float32)
        p = WanPipeline.from_pretrained(VID_MODEL, vae=vae, torch_dtype=torch.float16)
        try:
            p.enable_model_cpu_offload(); p.enable_vae_slicing()
        except Exception:
            p.to('cuda')
        VIDEO['pipe'] = p
        print('[+] video pipeline ready')
        return p
    except Exception as e:
        VIDEO['error'] = '%s: %s' % (type(e).__name__, e)
        raise RuntimeError(VIDEO['error'])
    finally:
        VIDEO['loading'] = False

def _is_oom(e):
    s = str(e).lower()
    return 'out of memory' in s or 'cuda oom' in s

def render_image(d):
    import torch
    p = (d.get('prompt') or '').strip()
    if p and not p.startswith('masterpiece'):
        p = 'masterpiece, best quality, score_7, ' + p
    neg = d.get('negative') or ('worst quality, low quality, score_1, score_2, score_3, '
                                'artist name, blurry, jpeg artifacts, chromatic aberration')
    if HAS_GPU:
        if pipe is None:
            raise RuntimeError('run cell 3 first')
        w = int(d.get('width') or 640); h = int(d.get('height') or 960)
        steps = int(d.get('steps') or 22); seed = int(d.get('seed') or 0)
        err = None
        for attempt in range(3):
            try:
                out = pipe(prompt=p, negative_prompt=neg, height=h, width=w,
                           num_inference_steps=steps,
                           generator=torch.Generator('cuda').manual_seed(seed))
                img = out.images[0]
                m = float(np.asarray(img.convert('RGB')).mean())
                if m < 6:
                    raise RuntimeError('black image (mean=%.1f) - rerun cell 3' % m)
                buf = io.BytesIO(); img.save(buf, format='PNG')
                return buf.getvalue(), 'png'
            except Exception as e:
                err = e
                if _is_oom(e) and (w > 512 or h > 512):
                    import gc; gc.collect(); torch.cuda.empty_cache()
                    w = max(512, (w*4//5)//8*8); h = max(512, (h*4//5)//8*8)
                    print('[!] OOM -> 降到 %dx%d 重试' % (w, h))
                    continue
                raise
        raise err
    else:
        if d.get('cpu_preset'):
            set_cpu_preset(str(d['cpu_preset']))
        pr = CPU_PRESETS[CPU_PRESET]; ov = load_cpu_pipe()
        dim = pr['maxdim']
        w = min(int(d.get('width') or dim), dim); h = min(int(d.get('height') or dim), dim)
        w -= w % 8; h -= h % 8
        kw = dict(prompt=p, height=h, width=w, num_inference_steps=pr['steps'],
                  guidance_scale=pr['guidance'],
                  generator=torch.Generator().manual_seed(int(d.get('seed') or 0)))
        kw.update(pr.get('extra') or {})
        if pr.get('neg'):
            kw['negative_prompt'] = neg
        img = ov(**kw).images[0]
        buf = io.BytesIO(); img.save(buf, format='PNG')
        return buf.getvalue(), 'png'

def render_video(d):
    import torch, subprocess
    from diffusers.utils import export_to_video, load_image
    vp = load_video_pipe()
    neg = d.get('negative') or ('模糊不清，字幕，最差质量，低质量，JPEG压缩残留，多余的手指')
    kw = dict(prompt=d.get('prompt') or '', negative_prompt=neg,
              height=int(d.get('height') or 704), width=int(d.get('width') or 1280),
              num_frames=int(d.get('frames') or 121), guidance_scale=float(d.get('guidance') or 5.0),
              num_inference_steps=int(d.get('steps') or 50),
              generator=torch.Generator('cuda').manual_seed(int(d.get('seed') or 0)))
    src = d.get('image_path') or ''
    if src and os.path.exists(src):
        kw['image'] = load_image(src)
    frames = vp(**kw).frames[0]
    raw = os.path.join(OUT_DIR, 'v_%d_raw.mp4' % int(time.time()))
    export_to_video(frames, raw, fps=int(d.get('fps') or 24))
    final = raw.replace('_raw.mp4', '.mp4')
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',raw,'-c:v','libx264','-crf','23',
                    '-preset','fast','-movflags','+faststart',final], check=True)
    try: os.remove(raw)
    except OSError: pass
    return open(final,'rb').read(), 'mp4'

def worker():
    while True:
        jid, kind, d = TASK_Q.get()
        set_job(jid, status='running', kind=kind)
        t0 = time.time()
        try:
            blob, ext = (render_video(d) if kind == 'video' else render_image(d))
            path = os.path.join(OUT_DIR, '%s.%s' % (jid, ext))
            open(path, 'wb').write(blob)
            set_job(jid, status='done', seconds=round(time.time()-t0,1),
                    path=path, bytes=len(blob), ext=ext)
        except Exception as e:
            set_job(jid, status='error', error=('%s: %s' % (type(e).__name__, e))[:300])
        finally:
            TASK_Q.task_done()

def submit(kind, d):
    jid = uuid.uuid4().hex
    set_job(jid, status='queued', kind=kind)
    TASK_Q.put((jid, kind, d))
    return jid

@app.route('/health')
def health():
    return jsonify(ok=True, gpu=HAS_GPU, gpu_name=GPU_NAME, vram_gb=round(VRAM_GB,1),
                   pending=TASK_Q.qsize(), jobs=len(JOBS), out=OUT_DIR, anima_mode=ANIMA_MODE,
                   video_loaded=VIDEO['pipe'] is not None, video_error=VIDEO['error'],
                   cpu_ready=CPU['pipe'] is not None, cpu_error=CPU['error'],
                   cpu_preset=CPU_PRESET, cpu_presets=list(CPU_PRESETS))

@app.route('/cpu_presets')
def cpu_presets_ep():
    return jsonify(current=CPU_PRESET, presets={k: v['model'] for k, v in CPU_PRESETS.items()})

@app.route('/generate', methods=['POST'])
def generate():
    d = request.get_json(force=True)
    if not (d.get('prompt') or '').strip():
        return jsonify(error='prompt required'), 400
    kind = d.get('kind') or 'image'
    if kind not in ('image','video'):
        return jsonify(error='kind must be image|video'), 400
    if kind == 'video' and not HAS_GPU:
        return jsonify(error='video needs GPU'), 400
    return jsonify(id=submit(kind, d), queued=TASK_Q.qsize())

@app.route('/batch', methods=['POST'])
def batch():
    d = request.get_json(force=True)
    prompts = d.get('prompts') or []
    if not prompts:
        return jsonify(error='prompts required'), 400
    n = int(d.get('n_per_prompt') or 1)
    kind = d.get('kind') or 'image'
    ids = []
    for pr in prompts:
        for k in range(n):
            dd = dict(d); dd['prompt'] = pr
            if d.get('seed'):
                dd['seed'] = int(d['seed']) + k
            ids.append(submit(kind, dd))
    return jsonify(ids=ids, count=len(ids), queued=TASK_Q.qsize())

@app.route('/status/<jid>')
def status(jid):
    j = get_job(jid)
    if not j:
        return jsonify(error='unknown id'), 404
    return jsonify(j)

@app.route('/image/<jid>')
@app.route('/result/<jid>')
def result(jid):
    j = get_job(jid)
    if not j or j.get('status') != 'done':
        return jsonify(error='not ready'), 404
    mime = 'video/mp4' if j.get('ext') == 'mp4' else 'image/png'
    return send_file(j['path'], mimetype=mime)

@app.route('/upload', methods=['POST'])
def upload():
    f = request.files.get('file')
    if not f:
        return jsonify(error='file field required'), 400
    name = re.sub(r'[^A-Za-z0-9_.-]', '_', f.filename or 'ref.png')
    path = os.path.join(OUT_DIR, 'up_' + name)
    f.save(path)
    return jsonify(ok=True, path=path)

@app.route('/flush')
def flush():
    flush_to_drive()
    return jsonify(ok=True, drive=DRIVE_OK)

threading.Thread(target=worker, daemon=True).start()
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=7860, debug=False, use_reloader=False),
                 daemon=True).start()

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget','-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O','/usr/local/bin/cloudflared'])
    os.chmod('/usr/local/bin/cloudflared', 0o755)
proc = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            URL = m.group(0); break

print('=' * 64)
print('PUBLIC URL:', URL)
print('=' * 64)
print("存到本机:  echo '%s' > ~/.dsh/.colab-url" % URL)
print('本机 CLI:  scripts/colab_gen.sh "prompt"')

In [ ]:
#@title 5. 自测一张（640x960@22，白衬衫同款配置）
import requests, time, io
from PIL import Image
r = requests.post(URL + '/generate', json={
    'prompt': '1girl, solo, silver long hair, blue eyes, white shirt, soft window light, detailed skin',
    'width': 640, 'height': 960, 'steps': 22, 'seed': 4242})
jid = r.json()['id']; print('job', jid)
t0 = time.time()
s = {}
while time.time() - t0 < 600:
    s = requests.get(URL + '/status/' + jid, timeout=30).json()
    st = s.get('status')
    if st in ('done', 'error'):
        print(st, s.get('seconds',''), (s.get('error') or '')[:150]); break
    time.sleep(5)
if s.get('status') == 'done':
    img = Image.open(io.BytesIO(requests.get(URL + '/result/' + jid, timeout=120).content))
    print('出图:', img.size, '%.0fs' % s.get('seconds', 0), '| mode=%s' % ANIMA_MODE)
    display(img)

In [ ]:
#@title 6. 批量出图（改 PROMPTS 一次提交，服务端排队串行出）
PROMPTS = [
    '1girl, solo, silver hair, standing in rain, city night, neon reflections',
    '1girl, solo, red kimono, cherry blossoms, sunlight, bokeh',
    '1girl, solo, black hair, reading in library, warm lamp light',
]
N_PER_PROMPT = 2
W, H, STEPS, SEED = 640, 960, 22, 1000

import requests, time
r = requests.post(URL + '/batch', json={'prompts': PROMPTS, 'n_per_prompt': N_PER_PROMPT,
                                        'width': W, 'height': H, 'steps': STEPS, 'seed': SEED})
ids = r.json()['ids']
per = 30 if HAS_GPU else 300
print('[+] 入队 %d 张，预计 ~%.0f 分钟' % (len(ids), len(ids)*per/60))
t0 = time.time()
while time.time() - t0 < len(ids)*per*4 + 600:
    st = [get_job(i) or {} for i in ids]
    d = sum(1 for s in st if s.get('status') == 'done')
    e = sum(1 for s in st if s.get('status') == 'error')
    print('  %d+%de/%d  %.1fmin' % (d, e, len(ids), (time.time()-t0)/60))
    if d + e >= len(ids):
        break
    time.sleep(10)
flush_to_drive()
print('[!] /content 会话结束清空，产出已搬 Drive')

In [ ]:
#@title 7. 视频（可选）：Wan 2.2 TI2V-5B，首次触发下 ~10-20GB 权重
MAKE_VIDEO = False
SRC_IMAGE = ''      # Colab 端静帧路径（cell 6 产出的 /content/out/xxx.png）；留空=文生视频
VID_PROMPT = 'gentle camera push in, hair swaying, soft light shift'
VID_FRAMES = 121    # 121帧@24fps=5秒；49 帧省一半时间
VID_STEPS = 50

if not MAKE_VIDEO:
    print('[i] MAKE_VIDEO=False，跳过')
elif not HAS_GPU:
    print('[!] 视频需要 GPU')
else:
    import requests, time, os
    payload = {'kind':'video','prompt':VID_PROMPT,'frames':VID_FRAMES,'steps':VID_STEPS,
               'height':704,'width':1280,'fps':24,'seed':42}
    if SRC_IMAGE and os.path.exists(SRC_IMAGE):
        payload['image_path'] = SRC_IMAGE
    r = requests.post(URL + '/generate', json=payload, timeout=60)
    jid = r.json()['id']; print('job', jid, '（首次先下权重）')
    t0 = time.time(); s = {}
    while time.time() - t0 < 2400:
        s = requests.get(URL + '/status/' + jid, timeout=30).json()
        if s.get('status') in ('done','error'):
            print(s.get('status'), s.get('seconds',''), (s.get('error') or '')[:150]); break
        time.sleep(15)
    if s.get('status') == 'done':
        mp4 = requests.get(URL + '/result/' + jid, timeout=300).content
        out = os.path.join(OUT_DIR, 'vid_%s.mp4' % jid[:8])
        open(out,'wb').write(mp4)
        print('[+]', out, '%.1fMB' % (len(mp4)/1024**2))
        flush_to_drive()

## 跑法与排错

**正常流程**：运行时改 GPU(T4) → 依次跑 1→2→3→4 → 出 URL 存本机 → 第 5 格自测 → 第 6 格批量。

**cell 3 输出怎么看**：
- `[hybrid] 测试图 mean=xxx -> 真图 OK` + `锁定 mode=hybrid` → 最快档成功，640×960 约 25-40s
- hybrid 黑图自动回退，`锁定 mode=fp32` → 能用但慢（~90-130s），属正常兜底
- 两种都失败 → 看报错重跑 cell 3（别跑别的格）

**常见问题**：
| 现象 | 处理 |
|---|---|
| `tunnel unreachable` / Cloudflare 1033 | runtime 重启过，重跑 cell 4，更新 ~/.dsh/.colab-url |
| 出图 `black image - rerun cell 3` | 精度模式失效，重跑 cell 3 |
| OOM | 已自动降档重试；还 OOM 就重跑 cell 3（清碎片）或降到 640×960 |
| RAM 爆 | 别反复重载模型；一个会话只跑一次 cell 3 |
| 排队慢 | 单 worker 是故意的（并发必 OOM），批量提交排队消化 |

**会话现实**：免费档 3.5-6h、/content 清空、12.6GB RAM。产出走 Drive 或本机下载。